# IMTalker **Training** on RunPod

**Efficient Audio-driven Talking Face Generation with Implicit Motion Transfer**

- Paper: https://arxiv.org/abs/2511.22167
- Model: https://huggingface.co/cbsjtu01/IMTalker
- Repo:  https://github.com/cbsjtu01/IMTalker

This notebook is the **training counterpart** of the inference notebook. It walks through:

1. GPU + environment check
2. Installing `conda` (if missing) and creating the `IMTalker` env (Python 3.10)
3. Installing PyTorch 2.3.1 (CUDA 12.1) + ffmpeg + project requirements
4. Cloning the IMTalker repo
5. Preparing the **renderer dataset** (`video_frame/` + `lmd/`)
6. Training the **renderer** (`renderer/train.py`)
7. Preparing the **generator dataset** (`motion/`, `audio/`, `smirk/`, `gaze/`)
   - Includes feature-extraction helpers for motion latents (from the trained renderer),
     Wav2Vec2 audio features, SMIRK 6D poses, and L2CS-Net gaze.
8. Training the **generator** (`generator/train.py`)
9. Launching **TensorBoard** to monitor losses
10. Resuming training / checkpoint management

> **README hardware reference**: training was done on **4 × A100 80 GB**.
> On a single 24 GB GPU (RTX 4090) you must lower `--batch_size` and probably also
> lower the learning rate. Expect long wall-clock times. RunPod multi-GPU pods are recommended for full training; this notebook is set up so a single-GPU pod still works for *fine-tuning* or small experiments.

> **Important — how cells run on RunPod**: A Jupyter notebook on RunPod uses the kernel that started Jupyter (usually the base `python`). `conda activate` does **not** persist across `!` shell cells, so every command that needs the `IMTalker` env is wrapped in `conda run -n IMTalker ...`.

In [ ]:
!pip install huggingface_hub
!pip install zstandard

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo_id = "Darknsu/HDTF_IMTalker_V1"
filename = "IMTalker-data.tar.zst"   # 👈 replace with actual file path inside repo
download_dir = "/workspace/all"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading selected file...")

file_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=filename,
    local_dir=download_dir,
    local_dir_use_symlinks=False,
)

print("✅ File downloaded to:", file_path)

In [ ]:
import os
import subprocess

tar_zst_path = "/workspace/all/IMTalker-data.tar.zst"

print("📦 Extracting .tar.zst file...")

# Step 1: decompress .zst → .tar
subprocess.run(["unzstd", "-f", tar_zst_path], check=True)

tar_path = tar_zst_path.replace(".zst", "")

print("📂 Extracting tar archive...")

# Step 2: extract tar
subprocess.run(["tar", "-xf", tar_path, "-C", os.path.dirname(tar_path)], check=True)

print("✅ Extraction complete!")

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo_id = "Darknsu/IMTalker_Step_1_Checkpoint"
filename = "last.ckpt"   # 👈 replace with actual file path inside repo
download_dir = "/workspace/"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading selected file...")

file_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename=filename,
    local_dir=download_dir,
    local_dir_use_symlinks=False,
)

print("✅ File downloaded to:", file_path)

In [ ]:
!git clone https://github.com/MoshiHead/IMTalker-training-v2-step2.git

## 0. Sanity check — GPU, CUDA, disk

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version || echo "nvcc not in PATH — that's OK, we install the CUDA 12.1 PyTorch wheels."
!df -h /workspace 2>/dev/null || df -h /

## 1. Choose a working directory

On RunPod, `/workspace` is the persistent volume — anything outside it disappears when the pod stops. Training checkpoints and TensorBoard logs are large, so we **must** keep them in `/workspace`.

In [ ]:
import os

WORKDIR = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

## 2. Install Miniconda (skip if already present)

Most RunPod PyTorch templates ship without conda. We install Miniconda silently into `/workspace/miniconda3`. If `conda` is already on the PATH the cell is essentially a no-op.

In [ ]:
%%bash
set -e
if command -v conda >/dev/null 2>&1; then
    echo "conda already installed at: $(which conda)"
    conda --version
    exit 0
fi

cd /workspace 2>/dev/null || cd ~
if [ ! -f Miniconda3-latest-Linux-x86_64.sh ]; then
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
fi
bash Miniconda3-latest-Linux-x86_64.sh -b -p "$PWD/miniconda3"
echo "Miniconda installed at $PWD/miniconda3"

In [ ]:
# Make `conda` available to subsequent ! cells by prepending its bin dir to PATH for this kernel.
import os, shutil, subprocess

candidates = [
    "/workspace/miniconda3/bin",
    os.path.expanduser("~/miniconda3/bin"),
    "/opt/conda/bin",
]
for c in candidates:
    if os.path.isfile(os.path.join(c, "conda")) and c not in os.environ["PATH"]:
        os.environ["PATH"] = c + ":" + os.environ["PATH"]

print("conda binary:", shutil.which("conda"))
subprocess.run(["conda", "--version"], check=True)

## 3. Create the `IMTalker` conda environment (Python 3.10)

Matches the README's `conda create -n IMTalker python=3.10`. If you already ran the **inference** notebook on this pod, the env will be reused — the cell is a no-op then.

In [ ]:
%%bash
set -e
# Accept Anaconda ToS just in case (newer conda versions require this before using defaults).
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main 2>/dev/null || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r    2>/dev/null || true

if conda env list | awk '{print $1}' | grep -qx "IMTalker"; then
    echo "Env 'IMTalker' already exists — skipping creation."
else
    conda create -y -n IMTalker -c conda-forge --override-channels python=3.10
fi
conda run -n IMTalker python --version

## 4. Install PyTorch 2.3.1 + cu121 and ffmpeg

These are the exact versions the README specifies. Skip if you already ran the inference notebook.

In [ ]:
%%bash
set -e

# 1. Install pip into the IMTalker env (conda-forge's python doesn't include it by default).
conda install -y -n IMTalker -c conda-forge --override-channels pip

# 2. Verify pip is now available inside the env.
conda run -n IMTalker python -m pip --version

# 3. Upgrade pip using the env's own interpreter.
conda run -n IMTalker python -m pip install --upgrade pip

# 4. Install PyTorch 2.3.1 + cu121 (exact versions from the README).
conda run -n IMTalker python -m pip install \
    torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 \
    --index-url https://download.pytorch.org/whl/cu121

# 5. Sanity check.
conda run -n IMTalker python -c "import torch, sys; print('python:', sys.version.split()[0]); print('torch :', torch.__version__); print('cuda  :', torch.cuda.is_available())"

In [ ]:
%%bash
set -e
# ffmpeg via conda-forge (matches README)
conda install -y -n IMTalker -c conda-forge --override-channels ffmpeg
conda run -n IMTalker ffmpeg -version | head -n 1

## 5. Clone the IMTalker repository

We clone the **official** IMTalker repo (which contains `renderer/train.py` and `generator/train.py`). If you already cloned an inference fork in the same pod, you may need a separate directory.

In [ ]:
%%bash
set -e
cd /workspace 2>/dev/null || cd ~
if [ ! -d IMTalker ]; then
    git clone https://github.com/cbsjtu01/IMTalker.git
else
    echo "IMTalker repo already present — pulling latest."
    cd IMTalker && git pull --ff-only || true
fi
ls IMTalker | head

In [ ]:
import os

REPO_DIR = "/workspace/IMTalker" if os.path.isdir("/workspace/IMTalker") else os.path.expanduser("~/IMTalker")
os.chdir(REPO_DIR)
print("Now in:", os.getcwd())
print("Has renderer/train.py :", os.path.exists(os.path.join(REPO_DIR, "renderer", "train.py")))
print("Has generator/train.py:", os.path.exists(os.path.join(REPO_DIR, "generator", "train.py")))

## 6. Install project Python requirements

Uses the repo's `requirement.txt` (singular — no `s`). Skip if you already ran the inference notebook in this env.

In [ ]:
%%bash
set -e
cd "${REPO_DIR:-/workspace/IMTalker}"
REQ_FILE="requirement.txt"
[ -f "$REQ_FILE" ] || REQ_FILE="requirements.txt"
echo "Using $REQ_FILE"
conda run -n IMTalker pip install -r "$REQ_FILE"

## 7. (Optional) Download pretrained checkpoints

You don't strictly need pretrained checkpoints to train from scratch, but two are useful:

- **`wav2vec2-base-960h`** — required for extracting audio features for the generator dataset.
- **`renderer.ckpt`** — needed if you want to **skip training the renderer** and only train the generator. It's also needed by the motion-encoder during generator-dataset extraction.

Run this cell to grab them into `./checkpoints`.

In [ ]:
!conda run -n IMTalker pip install -U huggingface_hub

In [ ]:
%%bash
set -e
cd "${REPO_DIR:-/workspace/IMTalker}"
mkdir -p checkpoints

conda run -n IMTalker hf download cbsjtu01/IMTalker \
    --repo-type model \
    --local-dir checkpoints

echo
echo "== checkpoints tree =="
ls -lh checkpoints
echo
ls -lh checkpoints/wav2vec2-base-960h 2>/dev/null || \
    echo "(wav2vec2 folder will appear once HF mirror sync is done)"

---
# Stage 1 — Train the Renderer

## 8. Prepare the renderer dataset

The README expects:

```
/path/to/renderer_dataset
├── video_frame
│   ├── video_0001
│   │   ├── image_001.jpg
│   │   ├── image_002.jpg
│   │   └── ...
│   ├── video_0002
│   └── ...
└── lmd
    ├── video_0001.txt
    ├── video_0002.txt
    └── ...
```

- **Resolution**: 512 × 512, face centered.
- **Frames**: extracted at native video FPS (typically 25).
- **Landmarks**: one `.txt` per video; one row per frame.

For producing this from raw videos, the README points to **[talkingfaceprocess](https://github.com/liutaocode/talking_face_preprocessing)**. We clone it below so you can run its pipeline inside the same pod.

In [ ]:
# === EDIT THESE TO POINT AT YOUR OWN DATA =============================
RENDERER_DATASET = "/workspace/data/renderer_dataset"   # the path above
RAW_VIDEO_DIR    = "/workspace/data/raw_videos"         # your .mp4 inputs (if any)
# ======================================================================

import os
os.makedirs(RENDERER_DATASET, exist_ok=True)
os.makedirs(os.path.join(RENDERER_DATASET, "video_frame"), exist_ok=True)
os.makedirs(os.path.join(RENDERER_DATASET, "lmd"), exist_ok=True)
os.makedirs(RAW_VIDEO_DIR, exist_ok=True)

print("Dataset root:", RENDERER_DATASET)
print("Has video_frame/:", os.path.isdir(os.path.join(RENDERER_DATASET, 'video_frame')))
print("Has lmd/        :", os.path.isdir(os.path.join(RENDERER_DATASET, 'lmd')))

### 8a. (Optional) Clone the preprocessing pipeline

If you already have a prepared dataset in the layout above, **skip this section**. Otherwise the cell below clones `talking_face_preprocessing` for you. Read its README before running its scripts — it has its own dependencies (some heavy: `mediapipe`, `face_alignment`, etc.).

In [ ]:
# %%bash
# set -e
# cd /workspace 2>/dev/null || cd ~
# if [ ! -d talking_face_preprocessing ]; then
#     git clone https://github.com/liutaocode/talking_face_preprocessing.git
# else
#     echo "talking_face_preprocessing already cloned."
# fi
# ls talking_face_preprocessing | head

### 8b. Quick dataset sanity check

Counts how many video folders, frames, and landmark files you have, and previews the first landmark file so you can confirm the format.

In [ ]:
# import os, glob

# video_frame_dir = os.path.join(RENDERER_DATASET, "video_frame")
# lmd_dir         = os.path.join(RENDERER_DATASET, "lmd")

# video_folders = sorted([d for d in glob.glob(os.path.join(video_frame_dir, "*")) if os.path.isdir(d)])
# lmd_files     = sorted(glob.glob(os.path.join(lmd_dir, "*.txt")))

# print(f"video folders : {len(video_folders)}")
# print(f"landmark files: {len(lmd_files)}")

# if video_folders:
#     sample = video_folders[0]
#     frames = sorted(glob.glob(os.path.join(sample, "*.jpg"))) + \
#              sorted(glob.glob(os.path.join(sample, "*.png")))
#     print(f"frames in {os.path.basename(sample)}: {len(frames)}")

# if lmd_files:
#     print("\n--- head of", os.path.basename(lmd_files[0]), "---")
#     with open(lmd_files[0]) as f:
#         for i, line in enumerate(f):
#             if i >= 3: break
#             print(line.rstrip())

## 9. Train the renderer

README reference command:

```bash
python renderer/train.py \
    --dataset_path /path/to/renderer_dataset \
    --exp_name renderer_exp \
    --batch_size 4 \
    --iter 7000000 \
    --lr 1e-4
```

**Hardware reality check (from the README):**
- 4 × A100 80 GB, `batch_size 4` → ≤ 50 GB VRAM used, ~1 s/iter.
- 7 M iterations is the full schedule. You almost certainly want fewer for first runs.

**RunPod tips:**
- On a single 24 GB GPU, start with `--batch_size 1` or `2`.
- Use `nohup` / `tmux` for true long runs — a notebook cell will keep training while the cell is busy, but if your browser disconnects you can't easily check on it. The cell below is for short / interactive runs.
- Checkpoints and logs go inside the repo by default (look for `exp/<exp_name>` or `logs/<exp_name>` depending on the repo version) — inspect `renderer/train.py` to confirm the exact output path and adapt `TENSORBOARD_LOGDIR` in section 11 accordingly.

In [ ]:
!conda run -n IMTalker pip install setuptools==69.5.1

In [ ]:
!conda run -n IMTalker pip install tensorboard

In [ ]:
!conda run -n IMTalker python -c "import pkg_resources; print('pkg_resources OK')"

In [ ]:
import subprocess

subprocess.Popen(
    "conda run -n IMTalker tensorboard "
    "--logdir /workspace/IMTalker/exps/renderer_step2 "
    "--port 6006 --bind_all",
    shell=True
)

print("TensorBoard started at port 6006")

In [ ]:
!rm -r /workspace/IMTalker/exps

In [ ]:
!CUDA_VISIBLE_DEVICES=0,1,2,3,4 PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python renderer/train.py \
  --dataset_path /workspace/IMTalker/data/renderer_dataset/ \
  --exp_name renderer_step2 \
  --exp_path ./exps \
  --resume_ckpt /workspace/IMTalker/checkpoints/last.ckpt \
  --batch_size 4 \
  --lr 5e-5 \
  --loss_pose 0.5 \
  --loss_head_stable 1.0 \
  --display_freq 2000 \
  --save_freq 5000


In [ ]:
# !CUDA_VISIBLE_DEVICES=0,1,2,3,4 PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python renderer/train.py \
#   --dataset_path /workspace/IMTalker/data/renderer_dataset/ \
#   --exp_name renderer_step2 \
#   --exp_path ./exps \
#   --resume_ckpt /workspace/IMTalker/checkpoints/last.ckptt \
#   --batch_size 4 \
#   --lr 5e-5 \
#   --loss_pose 0.5 \
#   --loss_head_stable 1.0 \
#   --display_freq 2000 \
#   --save_freq 5000


In [ ]:
!ls /workspace/IMTalker/checkpoints/last.ckpt

In [ ]:
# # Example: 4-GPU pod — set as needed.
# !CUDA_VISIBLE_DEVICES=0,1 PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python renderer/train.py \
#   --dataset_path /workspace/IMTalker/data/renderer_dataset/ \
#   --exp_name renderer_with_pose \
#   --exp_path ./exps \
#   --resume_ckpt /workspace/IMTalker/checkpoints/renderer.ckpt \
#   --batch_size 4 \
#   --lr 1e-4 \
#   --loss_pose 0.5


In [ ]:
# # === EDIT TRAINING HYPERPARAMS ========================================
# EXP_NAME    = "renderer_exp"
# BATCH_SIZE  = 1          # README uses 4 on 4×A100; lower for a 24 GB single-GPU pod
# ITERATIONS  = 200000     # README's full schedule is 7,000,000 — way too long for most pods
# LR          = 1e-4
# # ======================================================================
# print("Will train renderer with:")
# print(f"  dataset    : {RENDERER_DATASET}")
# print(f"  exp_name   : {EXP_NAME}")
# print(f"  batch_size : {BATCH_SIZE}")
# print(f"  iter       : {ITERATIONS}")
# print(f"  lr         : {LR}")

### 9a. Launch the training (foreground)

This blocks the cell until training finishes or you interrupt the kernel. For longer runs, see section 9b.

In [ ]:
# %cd /workspace/IMTalker
# !PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python renderer/train.py \
#     --dataset_path "{RENDERER_DATASET}" \
#     --exp_name "{EXP_NAME}" \
#     --batch_size {BATCH_SIZE} \
#     --iter {ITERATIONS} \
#     --lr {LR}

### 9b. Launch in the background with `nohup` (recommended for long runs)

The training process keeps running even if you close the browser or restart the kernel. Logs stream to `nohup_renderer.out`.

In [ ]:
# %%bash
# cd "${REPO_DIR:-/workspace/IMTalker}"
# mkdir -p logs
# nohup env PYTHONPATH="$PWD" conda run -n IMTalker --no-capture-output \
#     python renderer/train.py \
#         --dataset_path "$RENDERER_DATASET" \
#         --exp_name "$EXP_NAME" \
#         --batch_size $BATCH_SIZE \
#         --iter $ITERATIONS \
#         --lr $LR \
#     > logs/nohup_renderer.out 2>&1 &
# echo "PID: $!"
# echo "Tail with: !tail -f logs/nohup_renderer.out"

In [ ]:
# # Tail the live log (Ctrl+. / ■ to stop following — training keeps running)
# !tail -n 50 -f /workspace/IMTalker/logs/nohup_renderer.out

### 9c. Multi-GPU launch (if your pod has more than 1 GPU)

`renderer/train.py` is PyTorch-Lightning based according to `requirement.txt` (`pytorch-lightning==2.2.1`). On a multi-GPU pod, Lightning normally picks up all visible GPUs automatically. If the script exposes a `--gpus` / `--devices` flag, prefer it. Otherwise restrict GPUs with `CUDA_VISIBLE_DEVICES`.

In [ ]:
# Example: 4-GPU pod — set as needed.
# !CUDA_VISIBLE_DEVICES=0,1,2,3 PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python renderer/train.py \
#     --dataset_path "{RENDERER_DATASET}" \
#     --exp_name "{EXP_NAME}" \
#     --batch_size 4 \
#     --iter {ITERATIONS} \
#     --lr {LR}

---
# Stage 2 — Train the Generator

## 10. Prepare the generator dataset

This is the harder of the two stages, because the generator trains on **pre-extracted features** rather than raw videos. The README requires:

```
/path/to/generator_dataset
├── motion
│   ├── video_0001.pt    # motion latents from the renderer's motion encoder
│   ├── video_0002.pt
│   └── ...
├── audio
│   ├── video_0001.npy   # final-layer Wav2Vec2 features
│   ├── video_0002.npy
│   └── ...
├── smirk
│   ├── video_0001.pt    # 6D pose params, one row per frame
│   ├── video_0002.pt
│   └── ...
└── gaze
    ├── video_0001.npy   # gaze direction from L2CS-Net
    ├── video_0002.npy
    └── ...
```

You need:
1. **A trained renderer** (from Stage 1, or the downloaded `renderer.ckpt`) — for `motion/*.pt`.
2. **Wav2Vec2** (`wav2vec2-base-960h`) — for `audio/*.npy`.
3. **SMIRK** ([repo](https://github.com/georgeretsi/smirk)) — for `smirk/*.pt`.
4. **L2CS-Net** ([repo](https://github.com/Ahmednull/L2CS-Net)) — for `gaze/*.npy`.

Below are **starter helpers** for steps 1 and 2 (which can run inside the IMTalker env directly). SMIRK and L2CS-Net each have their own environments and weights; we just clone them and leave their extraction loops as TODOs you can adapt to your data.

In [ ]:
# # === EDIT PATHS ========================================================
# GENERATOR_DATASET = "/workspace/data/generator_dataset"
# RAW_AUDIO_DIR     = "/workspace/data/raw_audio"   # .wav files matching your video IDs
# TRAINED_RENDERER  = "/workspace/IMTalker/checkpoints/renderer.ckpt"  # or your own
# WAV2VEC2_DIR      = "/workspace/IMTalker/checkpoints/wav2vec2-base-960h"
# # =======================================================================

# import os
# for sub in ("motion", "audio", "smirk", "gaze"):
#     os.makedirs(os.path.join(GENERATOR_DATASET, sub), exist_ok=True)
# os.makedirs(RAW_AUDIO_DIR, exist_ok=True)
# print("Generator dataset root:", GENERATOR_DATASET)
# print("Renderer ckpt exists  :", os.path.exists(TRAINED_RENDERER))
# print("Wav2Vec2 dir exists   :", os.path.isdir(WAV2VEC2_DIR))

### 10a. Extract Wav2Vec2 audio features → `audio/*.npy`

For each `.wav` in `RAW_AUDIO_DIR`, we run the Wav2Vec2 base model and save the **final hidden state** as a `(T, 768)` float32 array. The filename matches the video ID.

> Adjust `target_sr = 16000` and the slicing logic to match how the IMTalker generator expects its audio features. Check `generator/train.py` and the generator's dataset class to confirm the exact tensor shape it loads.

In [ ]:
# # Run inside the IMTalker env so it picks up the right transformers / torch.
# # This writes a helper script to the repo and invokes it.
# EXTRACT_AUDIO_PY = '''
# import os, glob, numpy as np, torch, librosa, sys
# from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

# RAW_AUDIO_DIR   = sys.argv[1]
# OUT_DIR         = sys.argv[2]
# WAV2VEC2_DIR    = sys.argv[3]

# device = "cuda" if torch.cuda.is_available() else "cpu"
# print("Using:", device)

# fe    = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VEC2_DIR)
# model = Wav2Vec2Model.from_pretrained(WAV2VEC2_DIR).to(device).eval()

# os.makedirs(OUT_DIR, exist_ok=True)
# wavs = sorted(glob.glob(os.path.join(RAW_AUDIO_DIR, "*.wav")))
# print(f"Found {len(wavs)} wav files")

# with torch.no_grad():
#     for i, wav_path in enumerate(wavs):
#         name = os.path.splitext(os.path.basename(wav_path))[0]
#         out_path = os.path.join(OUT_DIR, name + ".npy")
#         if os.path.exists(out_path):
#             continue
#         audio, sr = librosa.load(wav_path, sr=16000, mono=True)
#         inputs = fe(audio, sampling_rate=16000, return_tensors="pt").input_values.to(device)
#         out = model(inputs).last_hidden_state.squeeze(0).cpu().numpy()  # (T, 768)
#         np.save(out_path, out.astype(np.float32))
#         if (i + 1) % 25 == 0 or i == len(wavs) - 1:
#             print(f"{i+1}/{len(wavs)}  {name}  shape={out.shape}")
# print("Done.")
# '''

# with open("/workspace/IMTalker/extract_audio_feats.py", "w") as f:
#     f.write(EXTRACT_AUDIO_PY)
# print("Wrote /workspace/IMTalker/extract_audio_feats.py")

In [ ]:
# !PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python /workspace/IMTalker/extract_audio_feats.py \
#     "{RAW_AUDIO_DIR}" \
#     "{GENERATOR_DATASET}/audio" \
#     "{WAV2VEC2_DIR}"

### 10b. Extract motion latents with the renderer → `motion/*.pt`

The renderer has a motion encoder; we feed each video's frames through it and save the per-frame latent stack as a `.pt` tensor.

> The exact class/module path of the motion encoder depends on the current `renderer/` source layout. The skeleton below shows the structure; you will need to import the right class and load weights from `TRAINED_RENDERER`. Check `renderer/inference.py` for the canonical usage — that file already loads the renderer and walks frames, so it's the best reference to adapt.

In [ ]:
# EXTRACT_MOTION_PY = '''
# """Skeleton: extract motion latents from the renderer's motion encoder.

# Adapt the imports + load logic to match the current repo layout. The
# renderer/inference.py file in the same repo is the canonical reference for
# how to build the model and prepare frames.
# """
# import os, glob, sys, torch
# from PIL import Image
# import torchvision.transforms as T

# # --- adapt these two lines to match the repo ---------------------------------
# # Example (PSEUDO-CODE — confirm against renderer/inference.py):
# # from renderer.model import Renderer
# # model = Renderer.load_from_checkpoint(sys.argv[3]).eval().cuda()
# # motion_encoder = model.motion_encoder
# raise NotImplementedError(
#     "Open renderer/inference.py, copy the model-construction + checkpoint-loading\n"
#     "code into this script, then expose the motion_encoder and run the loop below."
# )
# # -----------------------------------------------------------------------------

# VIDEO_FRAME_DIR = sys.argv[1]
# OUT_DIR         = sys.argv[2]
# CKPT            = sys.argv[3]

# os.makedirs(OUT_DIR, exist_ok=True)
# tf = T.Compose([T.Resize((512, 512)), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])

# videos = sorted([d for d in glob.glob(os.path.join(VIDEO_FRAME_DIR, "*")) if os.path.isdir(d)])
# print(f"Found {len(videos)} videos")

# with torch.no_grad():
#     for vi, vdir in enumerate(videos):
#         name = os.path.basename(vdir)
#         out_path = os.path.join(OUT_DIR, name + ".pt")
#         if os.path.exists(out_path):
#             continue
#         frames = sorted(glob.glob(os.path.join(vdir, "*.jpg")) +
#                         glob.glob(os.path.join(vdir, "*.png")))
#         latents = []
#         for fp in frames:
#             img = tf(Image.open(fp).convert("RGB")).unsqueeze(0).cuda()
#             z = motion_encoder(img).squeeze(0).cpu()
#             latents.append(z)
#         latents = torch.stack(latents, dim=0)
#         torch.save(latents, out_path)
#         if (vi + 1) % 10 == 0 or vi == len(videos) - 1:
#             print(f"{vi+1}/{len(videos)}  {name}  shape={tuple(latents.shape)}")
# print("Done.")
# '''

# with open("/workspace/IMTalker/extract_motion_feats.py", "w") as f:
#     f.write(EXTRACT_MOTION_PY)
# print("Wrote /workspace/IMTalker/extract_motion_feats.py — edit the marked section before running.")

In [ ]:
# Run AFTER you adapt the script (see comment block inside it).
# !PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python /workspace/IMTalker/extract_motion_feats.py \
#     "{RENDERER_DATASET}/video_frame" \
#     "{GENERATOR_DATASET}/motion" \
#     "{TRAINED_RENDERER}"

### 10c. SMIRK 6D pose features → `smirk/*.pt`

SMIRK has its own conda env and pretrained weights. Clone it once and follow its README's *inference* section to produce one `.pt` per video.

In [ ]:
%%bash
set -e
cd /workspace 2>/dev/null || cd ~
if [ ! -d smirk ]; then
    git clone https://github.com/georgeretsi/smirk.git
else
    echo "SMIRK already cloned."
fi
echo
echo "Next steps (do these manually following SMIRK's README):"
echo "  1) cd smirk && create its env, install deps, download its pretrained weights."
echo "  2) Run SMIRK on every video in $RENDERER_DATASET/video_frame ."
echo "  3) Save each result as a torch tensor of 6D pose params per frame to"
echo "     $GENERATOR_DATASET/smirk/<video_id>.pt ."

### 10d. L2CS-Net gaze direction → `gaze/*.npy`

Same idea: clone, follow upstream README, save per-frame gaze as `.npy`.

In [ ]:
%%bash
set -e
cd /workspace 2>/dev/null || cd ~
if [ ! -d L2CS-Net ]; then
    git clone https://github.com/Ahmednull/L2CS-Net.git
else
    echo "L2CS-Net already cloned."
fi
echo
echo "Next steps (do these manually following L2CS-Net's README):"
echo "  1) cd L2CS-Net && create its env, install deps, download its pretrained weights."
echo "  2) Run gaze estimation on every video in $RENDERER_DATASET/video_frame ."
echo "  3) Save each result as np.float32 of shape (T, 2) (pitch, yaw in radians) to"
echo "     $GENERATOR_DATASET/gaze/<video_id>.npy ."

### 10e. Sanity check the generator dataset

In [ ]:
import os, glob

subdirs = ["motion", "audio", "smirk", "gaze"]
ids = {}
for s in subdirs:
    files = glob.glob(os.path.join(GENERATOR_DATASET, s, "*"))
    names = {os.path.splitext(os.path.basename(f))[0] for f in files}
    ids[s] = names
    print(f"{s:6s}: {len(names):6d} files")

common = set.intersection(*ids.values()) if all(ids.values()) else set()
print(f"\nVideos with ALL four features: {len(common)}")
if ids["motion"] and ids["audio"]:
    only_motion = ids["motion"] - ids["audio"]
    only_audio  = ids["audio"]  - ids["motion"]
    print(f"  motion-only (no audio): {len(only_motion)}")
    print(f"  audio-only (no motion): {len(only_audio)}")

## 11. Train the generator

README reference command:

```bash
python generator/train.py \
    --dataset_pat /path/to/generator_dataset \
    --exp_name generator_exp \
    --batch_size 16 \
    --iter 5000000 \
    --lr 1e-4
```

> The README literally uses `--dataset_pat` (no final `h`). If the actual CLI in the repo source is `--dataset_path`, swap the flag — keep both versions of the cell below as a reference.

Hardware: 4 × A100 80 GB with `--batch_size 16` used ≤ 20 GB VRAM at ~10 iter/s and *converged in a few hours*. The generator is much lighter than the renderer.

In [ ]:
# === EDIT TRAINING HYPERPARAMS ========================================
GEN_EXP_NAME    = "generator_exp"
GEN_BATCH_SIZE  = 4         # README uses 16 across 4 GPUs; lower for single GPU
GEN_ITERATIONS  = 500000    # README's full schedule is 5,000,000
GEN_LR          = 1e-4
# ======================================================================
print("Will train generator with:")
print(f"  dataset    : {GENERATOR_DATASET}")
print(f"  exp_name   : {GEN_EXP_NAME}")
print(f"  batch_size : {GEN_BATCH_SIZE}")
print(f"  iter       : {GEN_ITERATIONS}")
print(f"  lr         : {GEN_LR}")

### 11a. Foreground launch

In [ ]:
%cd /workspace/IMTalker
# Try the flag name from the README first (--dataset_pat). If the script
# complains about an unknown argument, comment this cell out and use 11b.
!PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python generator/train.py \
    --dataset_pat "{GENERATOR_DATASET}" \
    --exp_name "{GEN_EXP_NAME}" \
    --batch_size {GEN_BATCH_SIZE} \
    --iter {GEN_ITERATIONS} \
    --lr {GEN_LR}

### 11b. Foreground launch (alternative flag name)

In [ ]:
%cd /workspace/IMTalker
# Use this version if the script uses --dataset_path (most likely the real name).
!PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python generator/train.py \
    --dataset_path "{GENERATOR_DATASET}" \
    --exp_name "{GEN_EXP_NAME}" \
    --batch_size {GEN_BATCH_SIZE} \
    --iter {GEN_ITERATIONS} \
    --lr {GEN_LR}

### 11c. Background launch with `nohup`

In [ ]:
%%bash
cd "${REPO_DIR:-/workspace/IMTalker}"
mkdir -p logs
nohup env PYTHONPATH="$PWD" conda run -n IMTalker --no-capture-output \
    python generator/train.py \
        --dataset_path "$GENERATOR_DATASET" \
        --exp_name "$GEN_EXP_NAME" \
        --batch_size $GEN_BATCH_SIZE \
        --iter $GEN_ITERATIONS \
        --lr $GEN_LR \
    > logs/nohup_generator.out 2>&1 &
echo "PID: $!"
echo "Tail with: !tail -f logs/nohup_generator.out"

In [ ]:
!tail -n 50 -f /workspace/IMTalker/logs/nohup_generator.out

## 12. Launch TensorBoard

`requirement.txt` already includes `tensorboard`. The exact log directory depends on what `renderer/train.py` / `generator/train.py` write to (usually `exp/<exp_name>/`, `logs/<exp_name>/`, or `lightning_logs/`). Inspect the repo source if in doubt.

On RunPod, expose port `6006` via **HTTP services** to access the dashboard from your browser.

In [ ]:
# Adjust this path if the training scripts write logs elsewhere.
TENSORBOARD_LOGDIR = "/workspace/IMTalker"
!ls {TENSORBOARD_LOGDIR}/exp 2>/dev/null || echo "(no exp/ dir yet — start training first)"
!ls {TENSORBOARD_LOGDIR}/logs 2>/dev/null || true
!ls {TENSORBOARD_LOGDIR}/lightning_logs 2>/dev/null || true

In [ ]:
# Long-running TensorBoard server. ■ to stop. Open via RunPod's port 6006 proxy URL.
!conda run -n IMTalker --no-capture-output tensorboard \
    --logdir "{TENSORBOARD_LOGDIR}" \
    --host 0.0.0.0 --port 6006

## 13. Checkpoint management / resuming

If `renderer/train.py` or `generator/train.py` expose a `--resume` or `--ckpt` flag (most Lightning-style trainers do), point it at your latest `.ckpt` to continue training. Inspect each script's `argparse` block to confirm the exact flag name.

In [ ]:
%%bash
# List the most recent checkpoints under the repo (depth-limited so it doesn't crawl forever).
cd "${REPO_DIR:-/workspace/IMTalker}"
find . -maxdepth 4 -name '*.ckpt' -printf '%T@ %p\n' 2>/dev/null \
    | sort -nr | head -n 20 | awk '{print strftime("%Y-%m-%d %H:%M", $1), $2}'

In [ ]:
# Example resume — adapt flag name (--resume / --ckpt / --resume_from_checkpoint) to your script.
# !PYTHONPATH=/workspace/IMTalker conda run -n IMTalker --no-capture-output python renderer/train.py \
#     --dataset_path "{RENDERER_DATASET}" \
#     --exp_name "{EXP_NAME}" \
#     --batch_size {BATCH_SIZE} \
#     --iter {ITERATIONS} \
#     --lr {LR} \
#     --resume /workspace/IMTalker/exp/renderer_exp/last.ckpt

## 14. (Optional) Copy final checkpoints somewhere safe

`/workspace` persists across pod restarts but is wiped when you delete the pod. If you want to keep a trained checkpoint for inference, copy it into a folder you'll back up, or upload to Hugging Face / S3 / GCS.

In [ ]:
%%bash
set -e
mkdir -p /workspace/trained_ckpts
# Edit the source paths once you know where the trainer writes them.
# cp /workspace/IMTalker/exp/renderer_exp/last.ckpt /workspace/trained_ckpts/renderer_mytrain.ckpt
# cp /workspace/IMTalker/exp/generator_exp/last.ckpt /workspace/trained_ckpts/generator_mytrain.ckpt
ls -lh /workspace/trained_ckpts || true

### 14a. Upload to Hugging Face (optional)

If you want to push the trained checkpoints to a private/public HF repo:

In [ ]:
# !conda run -n IMTalker huggingface-cli login
# !conda run -n IMTalker hf upload <YOUR_USERNAME>/<YOUR_REPO> /workspace/trained_ckpts --repo-type model

## 15. Troubleshooting cheat-sheet

- **`CUDA out of memory`** during renderer training — drop `--batch_size` to 1, lower input resolution if the script supports it, or rent a bigger pod. The README's 4 × A100 setup is the reference for full batch=4.
- **`ModuleNotFoundError`** — almost always means you ran a cell without `conda run -n IMTalker ...`. Don't drop that prefix.
- **`unrecognized arguments: --dataset_pat`** for the generator — the real flag in source is probably `--dataset_path` (the README typo). Use cell 11b.
- **Dataset loader is empty / silent hang at iter 0** — re-run the sanity-check cell in 10e; the four feature dirs likely don't share the same set of video IDs.
- **TensorBoard shows no scalars** — double-check `TENSORBOARD_LOGDIR`. Some Lightning configs write to `lightning_logs/` instead of `exp/`.
- **Browser disconnect kills training** — always use the `nohup` cells (9b, 11c) for runs longer than a few minutes.
- **Slow data loading on RunPod NVMe-less pods** — the renderer reads many small JPEGs. Either (a) pre-pack frames into a single tar/lmdb if the loader supports it, or (b) move the dataset to `/workspace` (persistent volume) rather than `/tmp`.
- **Resume from crash** — see section 13; the exact CLI flag depends on the version of the trainer in your repo.